# Native Apertus 2 messages

Run `make install` first, then open this notebook with the repository environment. Models are immutable; construction validates structure. Explicit `.check()` calls validate profile and conversation rules. These examples need no tokenizer artifact.


In [ ]:
from apertus_common import (
    Call,
    Conversation,
    Message,
    SystemPrompt,
    Thinking,
    ToolSpec,
    UnknownOutput,
    render_str,
)

weather = ToolSpec(
    name="weather",
    description="Current weather for one city.",
    json_schema={
        "type": "object",
        "properties": {"city": {"type": "string"}},
        "required": ["city"],
    },
)
system = SystemPrompt.build(behavior="Be concise.", tools=[weather])
assert system.thinking is Thinking.MEDIUM
# Direct construction requires thinking explicitly.
assert SystemPrompt(thinking=Thinking.LOW).thinking is Thinking.LOW

## Constructors and builders

Both return the same typed values. Builders never assign counters, insert waits, or reorder messages. Only `SystemPrompt` and `Call` also have one-shot `.build(...)` factories.


In [ ]:
call = Call.build(name="weather", counter=0, arguments={"city": "Zurich"})
assert call == (
    Call.builder().name("weather").counter(0).arguments({"city": "Zurich"}).build()
)
conversation = (
    Conversation.builder()
    .system(system)
    .host("Current date: 2026-10-01.")
    .document("Weather station notes.", from_="user", id="station-1")
    .user("What is the weather in Zurich?")
    .think("Consult the weather tool.")
    .message(call)
    .wait()
    .result(name="weather", counter=0, payload='{"tempC":18}')
    .claim("18 degrees Celsius")
    .reply("It is 18 degrees Celsius.")
    .wait()
    .build(check="full")
)
print([item.type for item in conversation.items])

## Native JSON and structural errors

Native storage includes `schema_version: 2`; message records carry a type discriminator; direction is a derived Python property. Loading preserves order and raw payloads. Native JSON uses `from` and `schema`; constructor aliases `from_` and `json_schema` are rejected during native loading. Known types reject extra fields and scalar coercion. Unknown outputs use `type="unknown_output"` and store their wire name in `output_type`.

`Conversation.from_json()` accepts only the current v2 schema. Older versions and direction-bearing records are rejected without migration. Optional `loss_weight` metadata belongs to v2, stays outside rendered content and requires opt-in profile checking.


In [ ]:
from pydantic import ValidationError

encoded = conversation.to_json()
restored = Conversation.from_json(encoded)
assert restored == conversation
assert Message.from_json(call.to_json()) == call
assert Conversation.model_json_schema()["$id"] == "urn:apertus-common:native:2"

unknown = UnknownOutput(
    output_type="memo", headers={"topic": "weather"}, payload="Extra."
)
assert Message.from_json(unknown.to_json()) == unknown

try:
    Call.model_validate({"name": "weather", "counter": True, "payload": "{}"})
except ValidationError as error:
    print("Boolean counter rejected:", error.errors()[0]["type"])
else:
    raise AssertionError("Expected structural validation to fail")

## Rendering

`render_str()` is a diagnostic view. Literal control spellings in payloads make text ambiguous. For generation, bind `load_encoding(artifact)` and use `encoding.encode_conversation(conversation)`, which starts with the tokenizer's BOS; token IDs preserve that distinction. Encoding does not run conformance checks.


In [ ]:
print(render_str(conversation))
assert conversation.check().ok
assert conversation.check().complete